# 00 Extract, Clean, and Split Manifest

Input Kaggle datasets:

- `/kaggle/input/datasets/tundng111/vimed-pet-ct-part1-raw-no-autoextract-v3`
- `/kaggle/input/datasets/tundng111/vimed-pet-ct-part2-raw-no-autoextract-v1`
- `/kaggle/input/datasets/hannhu4002/vimed-pet-part-3-archive`

Notebook này tự giải nén từng source base bằng `7z`, scan CT/PET/report,
kiểm tra file tồn tại và `.npy` readability, sau đó ghi clean manifest.

Gate tối thiểu cho bản full raw: `2,757` all cases, `2,757` clean/readable
cases nếu 3 archive đã đầy đủ, patient overlap = 0, val/test có đủ CT/PET/report.

Expected year-level counts from the ViMed-PET paper:

- 2017: 215 studies
- 2018: 462 studies
- 2019: 339 studies
- 2023: 1,741 studies
- Total: 2,757 studies

In [ ]:
from pathlib import Path
import os, re, json, shutil, subprocess, hashlib, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
MANIFEST_DIR = OUTPUT_ROOT / "00_manifest"
TEMP_ROOT = Path("/kaggle/temp/vimedpet_extract") if Path("/kaggle/temp").exists() else OUTPUT_ROOT / "_temp_extract"
STAGE_ROOT = Path("/kaggle/temp/vimedpet_archive_stage") if Path("/kaggle/temp").exists() else OUTPUT_ROOT / "_archive_stage"
for p in [MANIFEST_DIR, TEMP_ROOT, STAGE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

SOURCE_BASES = {
    "Part 1": ["PETCT_2017", "PETCT_2018", "PETCT_2019"],
    "Part 2": ["PETCT_2023_1", "PETCT_2023_2"],
    "Part 3": ["PETCT_2023_3"],
}
SOURCE_TO_PART = {src: part for part, items in SOURCE_BASES.items() for src in items}
EXPECTED_TOTAL_CASES = 2757
EXPECTED_YEAR_COUNTS = {
    "PETCT_2017": 215,
    "PETCT_2018": 462,
    "PETCT_2019": 339,
    "PETCT_2023": 1741,  # aggregate PETCT_2023_1 + PETCT_2023_2 + PETCT_2023_3
}
STRICT_FULL_DATASET_ASSERTS = True
INPUT_HINTS = [
    "/kaggle/input/datasets/tundng111/vimed-pet-ct-part1-raw-no-autoextract-v3",
    "/kaggle/input/datasets/tundng111/vimed-pet-ct-part2-raw-no-autoextract-v1",
    "/kaggle/input/datasets/hannhu4002/vimed-pet-part-3-archive",
    "/kaggle/input",
    str(Path.cwd()),
]

SPLIT_RATIOS = (0.70, 0.15, 0.15)
SPLIT_SEED = 391
QUICK_TEST_MAX_SOURCES = None  # đặt 1 nếu chỉ muốn test nhanh 1 source base

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("TEMP_ROOT:", TEMP_ROOT)

In [ ]:
def existing_roots():
    roots = []
    for x in INPUT_HINTS:
        p = Path(x)
        if p.exists():
            roots.append(p)
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    seen, out = set(), []
    for r in roots:
        rr = r.resolve()
        if rr not in seen:
            out.append(rr); seen.add(rr)
    return out

def find_archive_group(source_base):
    hits = []
    for root in existing_roots():
        for marker in list(root.rglob(f"{source_base}.zipraw")) + list(root.rglob(f"{source_base}.zip")):
            parts = sorted(marker.parent.glob(f"{source_base}.z[0-9][0-9]"))
            hits.append({"marker": marker, "parts": parts, "folder": marker.parent})
    if not hits:
        raise FileNotFoundError(f"Cannot find archive marker for {source_base}")
    hits = sorted(hits, key=lambda h: len(h["parts"]), reverse=True)
    return hits[0]

def stage_archive(source_base, group):
    stage = STAGE_ROOT / source_base
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)
    for part in group["parts"]:
        target = stage / part.name
        try:
            os.symlink(part, target)
        except Exception:
            shutil.copy2(part, target)
    zip_target = stage / f"{source_base}.zip"
    try:
        os.symlink(group["marker"], zip_target)
    except Exception:
        shutil.copy2(group["marker"], zip_target)
    return zip_target

def sevenzip_binary():
    for exe in ["7z", "7zz", "7za"]:
        try:
            subprocess.run([exe], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return exe
        except Exception:
            continue
    raise RuntimeError("7z/7zz/7za not found in this runtime")

def extract_source(source_base):
    extract_dir = TEMP_ROOT / source_base
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    group = find_archive_group(source_base)
    zip_path = stage_archive(source_base, group)
    exe = sevenzip_binary()
    cmd = [exe, "x", str(zip_path), f"-o{extract_dir}", "-y"]
    print("Extracting:", source_base, "parts:", len(group["parts"]), "marker:", group["marker"].name)
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-2000:])
    if result.returncode != 0:
        raise RuntimeError(f"7z failed for {source_base}")
    return extract_dir

def looks_like_petct_root(path):
    if not path.is_dir():
        return False
    return any((path / f"THANG {m}").exists() for m in range(1, 13))

def find_petct_root(extract_dir, source_base):
    candidates = [extract_dir / source_base, extract_dir / source_base / source_base, extract_dir]
    candidates += list(extract_dir.rglob(source_base))
    for c in candidates:
        if looks_like_petct_root(c):
            return c
    for c in extract_dir.rglob("*"):
        if looks_like_petct_root(c):
            return c
    raise FileNotFoundError(f"Cannot locate extracted PETCT root for {source_base}")

In [ ]:
def parse_day_patient(name):
    m = re.search(r"day[_-]?(\d+).*patient[_-]?(\d+)", name, flags=re.I)
    if not m:
        return None, None
    return int(m.group(1)), str(int(m.group(2)))

def read_json(path):
    for enc in ["utf-8", "utf-8-sig"]:
        try:
            return json.loads(Path(path).read_text(encoding=enc))
        except Exception:
            pass
    return {}

def flatten_json(obj, prefix=""):
    rows = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, (dict, list)):
                rows.update(flatten_json(v, key))
            else:
                rows[key] = v
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            rows.update(flatten_json(v, f"{prefix}[{i}]"))
    return rows

def choose_field(flat, patterns):
    for pat in patterns:
        for k, v in flat.items():
            if re.search(pat, k, flags=re.I):
                s = str(v).strip()
                if s and s.lower() != "nan":
                    return s
    return ""

def report_fields(report_path):
    obj = read_json(report_path)
    flat = flatten_json(obj)
    findings = choose_field(flat, [r"find", r"mô.?tả", r"kết.?quả", r"mo.?ta"])
    impression = choose_field(flat, [r"impression", r"kết.?luận", r"ket.?luan"])
    indication = choose_field(flat, [r"chỉ.?định", r"chi.?dinh", r"indication"])
    clinical_history = choose_field(flat, [r"lâm.?sàng", r"clinical", r"bệnh.?sử"])
    sex = choose_field(flat, [r"giới.?tính", r"sex"])
    birth_year = choose_field(flat, [r"năm.?sinh", r"birth"])
    text_parts = [findings, impression]
    if not any(text_parts):
        text_parts = [str(v) for v in flat.values() if isinstance(v, (str, int, float))]
    report_text = "\n".join([str(x).strip() for x in text_parts if str(x).strip()])
    return findings, impression, report_text, indication, clinical_history, sex, birth_year

def npy_readable(path):
    p = Path(path)
    if not p.exists():
        return False, "file not found", "", np.nan
    try:
        arr = np.load(p, mmap_mode="r")
        shape = tuple(int(x) for x in arr.shape)
        return True, "", str(shape), str(arr.dtype)
    except Exception as e:
        return False, str(e), "", ""

def scan_source(source_base, root):
    rows = []
    for month_dir in sorted(root.glob("THANG *")):
        if not month_dir.is_dir():
            continue
        m = re.search(r"\d+", month_dir.name)
        month = int(m.group(0)) if m else None
        ct_dir, pet_dir, rep_dir = month_dir / "CT", month_dir / "PET", month_dir / "reports"
        for ct_path in sorted(ct_dir.glob("*.npy")):
            stem = ct_path.stem
            pet_path = pet_dir / f"{stem}.npy"
            report_path = rep_dir / f"{stem}.json"
            day, patient_id = parse_day_patient(ct_path.name)
            ct_ok, ct_err, ct_shape, ct_dtype = npy_readable(ct_path)
            pet_ok, pet_err, pet_shape, pet_dtype = npy_readable(pet_path)
            findings = impression = report_text = indication = clinical_history = sex = birth_year = ""
            if report_path.exists():
                findings, impression, report_text, indication, clinical_history, sex, birth_year = report_fields(report_path)
            report_words = len(str(report_text).split())
            suv_mentions = len(re.findall(r"SUV", str(report_text), flags=re.I))
            case_key = f"{source_base}|{month_dir.name}|{stem}"
            text_empty = str(report_text).strip() == ""
            rows.append({
                "source_part": source_base,
                "dataset_part": SOURCE_TO_PART[source_base],
                "month": month,
                "day": day,
                "patient_id": patient_id,
                "global_patient_key": f"{source_base}|{patient_id}",
                "case_id": hashlib.sha1(case_key.encode()).hexdigest()[:12],
                "raw_case_key": case_key,
                "ct_img_path": str(ct_path.relative_to(root)),
                "pet_img_path": str(pet_path.relative_to(root)) if pet_path.exists() else str((pet_dir / f"{stem}.npy").relative_to(root)),
                "report_path": str(report_path.relative_to(root)),
                "ct_path": str(ct_path),
                "pet_path": str(pet_path),
                "report_path_resolved": str(report_path),
                "ct_exists": ct_path.exists(),
                "pet_exists": pet_path.exists(),
                "report_exists": report_path.exists(),
                "ct_readable": ct_ok,
                "pet_readable": pet_ok,
                "ct_error": ct_err,
                "pet_error": pet_err,
                "report_text_empty": text_empty,
                "ct_shape": ct_shape,
                "pet_shape": pet_shape,
                "ct_dtype": ct_dtype,
                "pet_dtype": pet_dtype,
                "findings": findings,
                "impression": impression,
                "report_text_clean": report_text,
                "target_text": f"Findings:\n{findings}\n\nImpression:\n{impression}".strip(),
                "report_word_count": report_words,
                "suv_mention_count": suv_mentions,
                "sex": sex,
                "birth_year": birth_year,
                "indication": indication,
                "clinical_history": clinical_history,
            })
    return pd.DataFrame(rows)

In [ ]:
all_frames = []
bad_frames = []
sources = [src for part in SOURCE_BASES.values() for src in part]
if QUICK_TEST_MAX_SOURCES is not None:
    sources = sources[:QUICK_TEST_MAX_SOURCES]

for source_base in sources:
    extract_dir = extract_source(source_base)
    petct_root = find_petct_root(extract_dir, source_base)
    df = scan_source(source_base, petct_root)
    all_frames.append(df)
    print(source_base, "rows:", len(df), "root:", petct_root)
    shutil.rmtree(extract_dir, ignore_errors=True)

all_cases = pd.concat(all_frames, ignore_index=True)
all_cases = all_cases.drop_duplicates(subset=["source_part", "ct_img_path", "pet_img_path", "report_path"]).reset_index(drop=True)
all_cases["source_year"] = all_cases["source_part"].str.extract(r"(PETCT_\d{4})", expand=False).fillna(all_cases["source_part"])
source_counts = all_cases["source_part"].value_counts().sort_index().rename_axis("source_part").reset_index(name="all_cases")
year_counts = all_cases["source_year"].value_counts().sort_index().rename_axis("source_year").reset_index(name="all_cases")
expected_rows = [
    {"source_year": k, "expected_cases": v, "computed_cases": int(year_counts.loc[year_counts.source_year == k, "all_cases"].sum())}
    for k, v in EXPECTED_YEAR_COUNTS.items()
]
expected_check = pd.DataFrame(expected_rows)
expected_check["status"] = np.where(expected_check["expected_cases"] == expected_check["computed_cases"], "OK", "CHECK")

issue_mask = (
    (~all_cases["ct_exists"]) | (~all_cases["pet_exists"]) | (~all_cases["report_exists"]) |
    (~all_cases["ct_readable"]) | (~all_cases["pet_readable"]) |
    (all_cases["report_text_empty"])
)
bad_records = all_cases[issue_mask].copy()
clean = all_cases[~issue_mask].copy().reset_index(drop=True)

def classify_issue(row):
    issues = []
    if not bool(row.get("ct_exists", False)): issues.append("missing_ct")
    if not bool(row.get("pet_exists", False)): issues.append("missing_pet")
    if not bool(row.get("report_exists", False)): issues.append("missing_report")
    if bool(row.get("ct_exists", False)) and not bool(row.get("ct_readable", False)): issues.append("ct_array_error")
    if bool(row.get("pet_exists", False)) and not bool(row.get("pet_readable", False)): issues.append("pet_array_error")
    if bool(row.get("report_text_empty", False)): issues.append("empty_report_text")
    return "|".join(issues) if issues else "ok"

if len(bad_records):
    bad_records["issue_type"] = bad_records.apply(classify_issue, axis=1)

rng = np.random.default_rng(SPLIT_SEED)
patients = np.array(sorted(clean["global_patient_key"].astype(str).unique()))
rng.shuffle(patients)
n = len(patients)
n_train = int(round(n * SPLIT_RATIOS[0]))
n_val = int(round(n * SPLIT_RATIOS[1]))
split_map = {}
for x in patients[:n_train]: split_map[x] = "train"
for x in patients[n_train:n_train+n_val]: split_map[x] = "validation"
for x in patients[n_train+n_val:]: split_map[x] = "test"
clean["clean_split"] = clean["global_patient_key"].map(split_map)
all_cases = all_cases.merge(clean[["case_id", "clean_split"]], on="case_id", how="left")
bad_records["clean_split"] = bad_records["global_patient_key"].map(split_map).fillna("excluded")

def overlap(df):
    sets = {s: set(df.loc[df.clean_split == s, "global_patient_key"].astype(str)) for s in ["train", "validation", "test"]}
    return {
        "train_validation": len(sets["train"] & sets["validation"]),
        "train_test": len(sets["train"] & sets["test"]),
        "validation_test": len(sets["validation"] & sets["test"]),
    }

summary = {
    "all_cases": int(len(all_cases)),
    "clean_cases": int(len(clean)),
    "bad_records": int(len(bad_records)),
    "expected_total_cases": int(EXPECTED_TOTAL_CASES),
    "matches_expected_total": bool(len(all_cases) == EXPECTED_TOTAL_CASES),
    "clean_matches_expected_total": bool(len(clean) == EXPECTED_TOTAL_CASES),
    "unique_patients": int(clean["global_patient_key"].nunique()),
    "split_counts": clean["clean_split"].value_counts().to_dict(),
    "patient_overlap": overlap(clean),
    "source_counts": clean["source_part"].value_counts().to_dict(),
    "all_source_counts": all_cases["source_part"].value_counts().to_dict(),
    "year_counts": all_cases["source_year"].value_counts().to_dict(),
    "part_counts": clean["dataset_part"].value_counts().to_dict(),
    "mean_report_words": float(clean["report_word_count"].mean()),
    "median_report_words": float(clean["report_word_count"].median()),
}

all_cases.to_csv(MANIFEST_DIR / "q1_all_cases_manifest.csv", index=False, encoding="utf-8-sig")
clean.to_csv(MANIFEST_DIR / "q1_clean_manifest.csv", index=False, encoding="utf-8-sig")
bad_records.to_csv(MANIFEST_DIR / "q1_bad_records.csv", index=False, encoding="utf-8-sig")
source_counts.to_csv(MANIFEST_DIR / "q1_source_counts.csv", index=False, encoding="utf-8-sig")
year_counts.to_csv(MANIFEST_DIR / "q1_year_counts.csv", index=False, encoding="utf-8-sig")
expected_check.to_csv(MANIFEST_DIR / "q1_expected_count_check.csv", index=False, encoding="utf-8-sig")
(MANIFEST_DIR / "q1_manifest_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

display(Markdown("### Manifest summary"))
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(Markdown("### Expected count check against ViMed-PET paper/card"))
display(expected_check)
display(Markdown("### All cases by source base"))
display(source_counts)
display(Markdown("### All cases by year"))
display(year_counts)
display(clean.groupby(["dataset_part", "source_part", "clean_split"]).size().rename("cases").reset_index())
display(Markdown("### Bad records"))
bad_cols = ["source_part", "case_id", "issue_type", "ct_exists", "pet_exists", "report_exists", "ct_readable", "pet_readable", "report_text_empty", "ct_error", "pet_error", "ct_path", "pet_path", "report_path_resolved"]
display(bad_records[[c for c in bad_cols if c in bad_records.columns]].head(50))
assert summary["patient_overlap"] == {"train_validation": 0, "train_test": 0, "validation_test": 0}
assert clean[clean.clean_split.isin(["validation", "test"])][["ct_exists", "pet_exists", "report_exists"]].all().all()
if STRICT_FULL_DATASET_ASSERTS:
    assert len(all_cases) == EXPECTED_TOTAL_CASES, f"Expected {EXPECTED_TOTAL_CASES} all cases, got {len(all_cases)}"
    assert (expected_check["status"] == "OK").all(), expected_check
    assert len(bad_records) == 0, f"Expected 0 bad records for full fixed raw dataset, got {len(bad_records)}"
    assert len(clean) == EXPECTED_TOTAL_CASES, f"Expected {EXPECTED_TOTAL_CASES} clean cases, got {len(clean)}"
print("Saved manifest folder:", MANIFEST_DIR)